# Instrument Findings, Question-Level Reference

Consumer file for instrument-shaped Findings (QS, FT, RS).

**Scope decisions** are documented in `cosmos-bc-dss/COSMoS_Instrument_Layer.md`. Summary:
- Scope: QS, FT, RS as classified in the Domain Pattern Inventory
- Two-sheet structure parallel to `Specimen_Findings.ipynb`:
  - Sheet 1 `Test_Identity`: TESTCD-grain, all green test codes scoped to instrument domains, with COSMoS BC summary attached where coverage exists
  - Sheet 2 `Instrument_Specs`: DSS-grain, one row per Dataset Specialization
- Green-side input: `SDTM_Instrument_Test_Identity.xlsx` (test grain), `SDTM_Instrument_Identity.xlsx` (codelist grain, C20993 + C211913 NCIt anchors), plus the RS subset of `SDTM_Test_Identity.xlsx` for the domain-level `RSTESTCD` codelist sub-pattern
- `CT_Sub_Pattern` column on Sheet 2 distinguishes per-instrument codelist rows, domain-level `RSTESTCD` rows (where 4 multi-framework TESTCDs fan out by criteria framework), and LZZT-example rows with no published CT identity
- Grouping exposure: multi-value `Categories` column on Sheet 1 and Sheet 2, normalised edge list on Sheet 3
- Deferred to v2: NCIt C211913 reference rows for the materialization gap (requires upstream artifact decision, see README)


In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('INSTRUMENT FINDINGS, QUESTION-LEVEL REFERENCE')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')

## 1. Setup


In [ ]:
BASE_DIR = Path.cwd().parent        # sdtm-findings/
REPO_ROOT = BASE_DIR.parent          # cdisc-for-ai/

GREEN_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Instrument_Test_Identity.xlsx'
GREEN_INSTRUMENT_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Instrument_Identity.xlsx'
GREEN_RS_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
YELLOW_FILE = REPO_ROOT / 'cosmos-bc-dss' / 'interim' / 'COSMoS_BC_DSS.xlsx'
DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'

MA_DIR = BASE_DIR / 'machine_actionable'
MA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = MA_DIR / 'Instrument_Findings.xlsx'

for f, label in [(GREEN_FILE, 'Green'), (GREEN_INSTRUMENT_FILE, 'Green Instr'), (GREEN_RS_FILE, 'Green RS'), (YELLOW_FILE, 'Yellow'), (DOMAIN_META_FILE, 'Domain Meta')]:
    if f.exists():
        print(f'  {label}: {f}')
    else:
        raise FileNotFoundError(f'{label} file not found: {f}')

print(f'  Output: {OUTPUT_FILE}')


## 2. Load Input Files


In [ ]:
green = pd.read_excel(GREEN_FILE, sheet_name='Test Codes', dtype=str).fillna('')
green_instrument = pd.read_excel(GREEN_INSTRUMENT_FILE, sheet_name='Instruments', dtype=str).fillna('')
green_rs = pd.read_excel(GREEN_RS_FILE, sheet_name='Test Codes', dtype=str).fillna('')
yellow = pd.read_excel(YELLOW_FILE, sheet_name='BC_DSS', dtype=str).fillna('')
domain_meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str)

print(f'Green (Instrument_Test_Identity): {len(green):,} rows x {len(green.columns)} columns')
print(f'Green (Instrument_Identity):      {len(green_instrument):,} rows x {len(green_instrument.columns)} columns')
print(f'Green (Test_Identity):            {len(green_rs):,} rows x {len(green_rs.columns)} columns')
print(f'Yellow: {len(yellow):,} rows x {len(yellow.columns)} columns')
print(f'Yellow row types: {yellow["Row_Type"].value_counts().to_dict()}')
print(f'Domain metadata: {len(domain_meta)} domains')


## 2b. Define Instrument Domain Scope


In [ ]:
# Instrument domain scope, from COSMoS behavioural analysis
# See COSMoS_Instrument_Layer.md and COSMoS_Behavioural_Analysis.md
#
# Three SDTM domains classified as Instrument Findings:
#   QS, Questionnaires
#   FT, Functional Tests
#   RS, Disease Response and Clinical Classification
#
# Behavioural pattern: BC = grouping (instrument question container),
# DSS layer is 1:1 with the BC (DS_Code = TESTCD). Decomposition axis is
# the hierarchy (instrument > subscale > question), not specimen or method.

INSTRUMENT_DOMAINS = {'QS', 'FT', 'RS'}

# Verify all in-scope domains exist in metadata and are flagged as QRS instrument
for d in sorted(INSTRUMENT_DOMAINS):
    row = domain_meta[domain_meta['Domain'] == d]
    if row.empty:
        raise ValueError(f'Domain {d} not found in metadata')
    notes = str(row['Notes'].iloc[0])
    obs = row['Observation_Class'].iloc[0]
    if 'QRS instrument' not in notes:
        raise ValueError(f'Domain {d} Notes={notes!r}, expected QRS instrument')
    print(f'  {d}: {obs} ({notes})')

print(f'\nInstrument domains in scope: {sorted(INSTRUMENT_DOMAINS)}')

## 3. Extract Instrument-Scope DSS Rows

Yellow-driven extraction of the 175 instrument-scope DSS rows with CT sub-pattern classification. Used by both Sheet 1 (aggregated) and Sheet 2 (one row per DSS).


In [ ]:
# Instrument-scope DSS rows, yellow-driven (one row per Dataset Specialization).
# Three CT codelist sub-patterns per COSMoS_Instrument_Layer.md section 4:
#   - Per-instrument codelist           TESTCD found in SDTM_Instrument_Test_Identity
#   - Domain-level RSTESTCD             RS-domain TESTCD found in SDTM_Test_Identity
#   - Not in published CT (LZZT)        no green identity in any published CT

dss_all = yellow[yellow['Row_Type'] == 'DSS'].copy()
dss_inst = dss_all[
    (dss_all['TESTCD'] != '') &
    (dss_all['Domain'].isin(INSTRUMENT_DOMAINS))
].copy()

print(f'All DSS rows: {len(dss_all):,}')
print(f'Instrument-domain DSS rows with TESTCD: {len(dss_inst):,}')
print(f'Domains in scope: {sorted(dss_inst["Domain"].unique())}')

inst_codes = set(green['TESTCD'].unique())
rs_codes = set(green_rs[
    green_rs['SDTM_Domains'].str.contains(r'\bRS\b', regex=True, na=False)
]['TESTCD'].unique())


def classify_ct_sub_pattern(row):
    if row['TESTCD'] in inst_codes:
        return 'Per-instrument codelist'
    if row['Domain'] == 'RS' and row['TESTCD'] in rs_codes:
        return 'Domain-level RSTESTCD'
    return 'Not in published CT (LZZT example)'


dss_inst['CT_Sub_Pattern'] = dss_inst.apply(classify_ct_sub_pattern, axis=1)

print()
print('CT sub-pattern distribution:')
for k, v in dss_inst['CT_Sub_Pattern'].value_counts().items():
    print(f'  {k}: {v}')


## 4. Build Sheet 1: Test_Identity

One row per TESTCD. Universe is the union of `SDTM_Instrument_Test_Identity.xlsx` (10,166 codes, all instrument domains) and the RS subset of `SDTM_Test_Identity.xlsx` (47 codes for the domain-level `RSTESTCD` codelist). Parallel to `Specimen_Findings.ipynb` Sheet 1.


In [ ]:
# === Sheet 1 universe: union of green test codes scoped to instrument domains ===
# Source 1: SDTM_Instrument_Test_Identity (10,166 rows, all in QS/FT/RS)
# Source 2: RS subset of SDTM_Test_Identity (47 rows, domain-level RSTESTCD codelist)
# These universes are disjoint; no TESTCD overlap.
#
# Both sources are single-domain in this scope (the 47 RS-tagged test codes
# from Test_Identity all have SDTM_Domains == 'RS'), so a single Domain
# column suffices for Sheet 1 - simpler than Specimen which has many
# cross-domain test codes.

TESTCODE_GREEN_COLS = [
    'NCIt_Code', 'TESTCD', 'TEST', 'Domain',
    # Instrument-layer identity (C20993 / C211913 anchors from SDTM_Instrument_Identity.xlsx)
    'Instrument', 'Codelist_TESTCD',
    'Instrument_NCIt_Code', 'Instrument_NCIt_Preferred_Term',
    'Container_NCIt_Code', 'Container_NCIt_Preferred_Term',
    # Test-level NCIt enrichment (from SDTM_Instrument_Test_Identity.xlsx)
    'NCIt_Preferred_Term', 'NCIt_Synonyms', 'NCIt_Definition',
    'UMLS_CUI', 'NCIm_CUI',
]

# Codelist-grain lookup: Codelist_TESTCD -> instrument & container NCIt anchors
CODELIST_LOOKUP_COLS = [
    'Codelist_TESTCD',
    'Instrument_NCIt_Code', 'Instrument_NCIt_Preferred_Term',
    'Container_NCIt_Code', 'Container_NCIt_Preferred_Term',
]
codelist_lookup = green_instrument[CODELIST_LOOKUP_COLS].drop_duplicates('Codelist_TESTCD')

# Instrument_Test_Identity has Codelist_TESTCD — left-join the codelist-grain NCIt anchors
green_inst_full = green.merge(
    codelist_lookup, on='Codelist_TESTCD', how='left',
).fillna('')
green_inst_full = green_inst_full[TESTCODE_GREEN_COLS].copy()

# Test_Identity RS subset: no Codelist_TESTCD, instrument-layer columns stay empty
green_rs_full = green_rs[green_rs['SDTM_Domains'].str.contains(
    r'\bRS\b', regex=True, na=False
)].copy()
green_rs_full['Domain'] = 'RS'
for col in ['Instrument', 'Codelist_TESTCD',
            'Instrument_NCIt_Code', 'Instrument_NCIt_Preferred_Term',
            'Container_NCIt_Code', 'Container_NCIt_Preferred_Term']:
    green_rs_full[col] = ''
green_rs_full = green_rs_full[TESTCODE_GREEN_COLS]

green_universe = pd.concat([green_inst_full, green_rs_full], ignore_index=True)
print(f'Test_Identity universe: {len(green_universe):,} test codes')
print(f'  From Instrument_Test_Identity: {len(green_inst_full):,}')
print(f'  From Test_Identity (RS):  {len(green_rs_full):,}')
print(f'  By domain: {green_universe["Domain"].value_counts().to_dict()}')


In [ ]:
# === COSMoS summary per TESTCD (Sheet 1 aggregation) ===
# Aggregates the 175 dss_inst rows by TESTCD. For per-instrument codelist
# test codes this is 1:1 (one DSS per TESTCD). For the four multi-framework
# RSTESTCDs (NEWLIND, NTRGRESP, OVRLRESP, TRGRESP) it rolls up the framework
# variants. COSMoS_DSS_Count signals the fan-out and points the user at
# Sheet 2 for the per-DSS detail.

def cosmos_summary_for_testcd(group):
    bc_names = group['BC_Name'].unique()
    bc_ids = group['COSMoS_BC_ID'].unique()
    sdtm_cats = group['COSMoS_Category'].unique()
    sdtm_subcats = group['COSMoS_Subcategory'].unique()
    categories = group['Categories'].unique()
    parents = group['Parent_Label'].unique()
    hierarchies = group['Hierarchy_Path'].unique()
    return pd.Series({
        'Has_COSMoS':         'Yes',
        'COSMoS_DSS_Count':   str(len(group)),
        'COSMoS_BC_ID':       '; '.join(b for b in bc_ids if b),
        'COSMoS_BC_Name':     '; '.join(n for n in bc_names if n),
        'COSMoS_Category':    '; '.join(c for c in sdtm_cats if c),
        'COSMoS_Subcategory': '; '.join(c for c in sdtm_subcats if c),
        'COSMoS_Categories':  '; '.join(c for c in categories if c),
        'COSMoS_Parent':      '; '.join(p for p in parents if p),
        'COSMoS_Hierarchy':   '; '.join(h for h in hierarchies if h),
    })

cosmos_summary = dss_inst.groupby('TESTCD').apply(cosmos_summary_for_testcd).reset_index()
print(f'COSMoS summary: {len(cosmos_summary)} TESTCDs with instrument-domain BC coverage')


In [ ]:
# === Build test_codes dataframe: green universe + COSMoS summary ===
test_codes = green_universe.merge(cosmos_summary, on='TESTCD', how='left').fillna('')
test_codes.loc[test_codes['Has_COSMoS'] == '', 'Has_COSMoS'] = 'No'
test_codes.loc[test_codes['COSMoS_DSS_Count'] == '', 'COSMoS_DSS_Count'] = '0'

# Sort for direct readability: Domain, then Has_COSMoS (Yes first so the
# COSMoS-covered subset is up front within each domain), then TESTCD.
HAS_COSMOS_ORDER = ['Yes', 'No']
test_codes['_hc_sort'] = pd.Categorical(
    test_codes['Has_COSMoS'], categories=HAS_COSMOS_ORDER, ordered=True
)
test_codes = test_codes.sort_values(
    by=['Domain', '_hc_sort', 'TESTCD'],
    kind='stable',
).drop(columns=['_hc_sort']).reset_index(drop=True)

n_with = (test_codes['Has_COSMoS'] == 'Yes').sum()
n_without = (test_codes['Has_COSMoS'] == 'No').sum()
print(f'Test_Identity: {len(test_codes):,} rows')
print(f'  With COSMoS BC coverage:    {n_with:,} ({n_with/len(test_codes)*100:.2f}%)')
print(f'  Without COSMoS BC coverage: {n_without:,} ({n_without/len(test_codes)*100:.2f}%)')


## 5. Build Sheet 2: Instrument_Specs

One row per Dataset Specialization. Yellow-driven; green NCIt enrichment attached per row from whichever green source matches the TESTCD. The four multi-framework TESTCDs (NEWLIND, NTRGRESP, OVRLRESP, TRGRESP) produce one row per criteria framework. Parallel to `Specimen_Findings.ipynb` Sheet 2.


In [ ]:
# Attach green NCIt enrichment per DSS row.
# Yellow already carries NCIt_Code (test concept) and TEST for all 175 rows;
# green only contributes the enrichment columns that yellow does not have:
# Instrument, Codelist_TESTCD, NCIt_Preferred_Term, NCIt_Synonyms,
# NCIt_Definition, UMLS_CUI, NCIm_CUI.
#
# Source 1: SDTM_Instrument_Test_Identity (per-instrument codelist rows)
# Source 2: RS subset of SDTM_Test_Identity (domain-level RSTESTCD rows)
# LZZT rows have no published CT identity in either source and stay unjoined.
#
# Instrument and Codelist_TESTCD are absent from Test_Identity, so they
# stay empty for the 21 RSTESTCD rows. Domain comes from the DSS row.

GREEN_JOIN_COLS = [
    'TESTCD',  # join key
    # Instrument-layer identity (C20993 / C211913 anchors)
    'Instrument', 'Codelist_TESTCD',
    'Instrument_NCIt_Code', 'Instrument_NCIt_Preferred_Term',
    'Container_NCIt_Code', 'Container_NCIt_Preferred_Term',
    # Test-level NCIt enrichment
    'NCIt_Preferred_Term', 'NCIt_Synonyms', 'NCIt_Definition',
    'UMLS_CUI', 'NCIm_CUI',
]

# Instrument_Test_Identity + codelist-grain NCIt anchors
green_inst_join = green.merge(
    codelist_lookup, on='Codelist_TESTCD', how='left',
).fillna('')
green_inst_join = green_inst_join[GREEN_JOIN_COLS].copy()

green_rs_scoped = green_rs[green_rs['SDTM_Domains'].str.contains(
    r'\bRS\b', regex=True, na=False
)].copy()
for col in ['Instrument', 'Codelist_TESTCD',
            'Instrument_NCIt_Code', 'Instrument_NCIt_Preferred_Term',
            'Container_NCIt_Code', 'Container_NCIt_Preferred_Term']:
    green_rs_scoped[col] = ''
green_rs_join = green_rs_scoped[GREEN_JOIN_COLS]

# A TESTCD present in both sources is preferred from Instrument_Test_Identity
# (Instrument and Codelist_TESTCD populated there).
green_combined = pd.concat([green_inst_join, green_rs_join], ignore_index=True)
green_combined = green_combined.drop_duplicates(subset='TESTCD', keep='first')

# Yellow-driven left join: 175 DSS rows in, 175 rows out, no column collisions.
specs = dss_inst.merge(
    green_combined,
    on='TESTCD',
    how='left',
).fillna('')

# Sort for direct readability: Domain, then CT sub-pattern (per-instrument
# first, then RSTESTCD fan-out, then LZZT data quality flag), then TESTCD,
# then framework so multi-framework rows appear in deterministic order.
CT_PATTERN_ORDER = [
    'Per-instrument codelist',
    'Domain-level RSTESTCD',
    'Not in published CT (LZZT example)',
]
specs['_ct_sort'] = pd.Categorical(
    specs['CT_Sub_Pattern'], categories=CT_PATTERN_ORDER, ordered=True
)
specs = specs.sort_values(
    by=['Domain', '_ct_sort', 'TESTCD', 'COSMoS_Subcategory'],
    kind='stable',
).drop(columns=['_ct_sort']).reset_index(drop=True)

with_enrich = (specs['NCIt_Preferred_Term'] != '').sum()
without_enrich = (specs['NCIt_Preferred_Term'] == '').sum()
print(f'Instrument_Specs: {len(specs):,} rows (yellow-driven)')
print(f'  With green enrichment:    {with_enrich}')
print(f'  Without green enrichment: {without_enrich}  (LZZT example artifacts)')


## 6. Build Sheet 3: Instrument_Grouping

Normalised edge list of `Categories` tokens. One row per (BC, Category) pair. Built from yellow only, since `Categories` is a COSMoS-side field. Sheet 1 and Sheet 2 carry the same data as a multi-value column for locality; Sheet 3 makes it queryable without parsing.


In [ ]:
# Build edge list: one row per (BC, Category token) for instrument scope
edges = []
for _, r in dss_inst.iterrows():
    cats = [c.strip() for c in str(r['Categories']).split(';') if c.strip()]
    for cat in cats:
        edges.append({
            'BC_Name':       r['BC_Name'],
            'BC_NCIt_Code':  r['NCIt_Code'],
            'TESTCD':        r['TESTCD'],
            'Domain':        r['Domain'],
            'Category':      cat,
        })

groupings = pd.DataFrame(edges).drop_duplicates().reset_index(drop=True)
print(f'Instrument_Grouping: {len(groupings):,} edges')
print(f'  Distinct BCs:        {groupings["BC_Name"].nunique()}')
print(f'  Distinct categories: {groupings["Category"].nunique()}')

## 7. Coverage Analysis


In [ ]:
print('=== Coverage ===')
print()
print('Sheet 1 (Test_Identity, TESTCD-grain):')
print(f'  Total test codes (green universe): {len(test_codes):,}')
print(f'    With COSMoS BC coverage:    {n_with:,} ({n_with/len(test_codes)*100:.2f}%)')
print(f'    Without COSMoS BC coverage: {n_without:,} ({n_without/len(test_codes)*100:.2f}%)')
print()
print('  Per domain:')
for d in sorted(INSTRUMENT_DOMAINS):
    sub = test_codes[test_codes['Domain'] == d]
    sub_yes = (sub['Has_COSMoS'] == 'Yes').sum()
    pct = sub_yes/len(sub)*100 if len(sub) else 0
    print(f'    {d}: {len(sub):>6,} test codes, {sub_yes:>4,} with COSMoS coverage ({pct:.2f}%)')
print()

print('Sheet 2 (Instrument_Specs, DSS-grain):')
print(f'  Total DSS rows (yellow): {len(specs):,}')
print()
print('  By CT sub-pattern:')
for k, v in specs['CT_Sub_Pattern'].value_counts().items():
    print(f'    {k}: {v}')
print()
print('  By domain:')
for d in sorted(INSTRUMENT_DOMAINS):
    sub = specs[specs['Domain'] == d]
    print(f'    {d}: {len(sub):>4,} DSS rows')
print()
print('  Domain x CT sub-pattern crosstab:')
print(pd.crosstab(specs['Domain'], specs['CT_Sub_Pattern']))
print()

# Multi-framework fan-out (RS classification-framework sub-pattern)
multi_fw = specs.groupby('TESTCD').size()
multi_fw = multi_fw[multi_fw > 1].sort_index()
print(f'  TESTCDs with >1 DSS row (RS classification-framework fan-out): {len(multi_fw)}')
for tc, n in multi_fw.items():
    fws = sorted(specs[specs['TESTCD'] == tc]['COSMoS_Subcategory'].unique())
    print(f'    {tc}: {n} rows, frameworks: {fws}')
print()
print(f'Distinct COSMoS BCs in scope: {dss_inst["BC_Name"].nunique()}')


## 8. Write Output


In [ ]:
# Sheet 1 column definitions (Test_Identity, one row per TESTCD)
TESTCODE_COLS = [
    # Green identity — test level
    ('NCIt_Code',           12),
    ('TESTCD',              12),
    ('TEST',                45),
    ('Domain',              8),
    # Instrument-layer identity (C20993 / C211913 anchors)
    ('Instrument',          50),
    ('Codelist_TESTCD',     16),
    ('Instrument_NCIt_Code',            14),
    ('Instrument_NCIt_Preferred_Term',  45),
    ('Container_NCIt_Code',             14),
    ('Container_NCIt_Preferred_Term',   45),
    # Green identity — test NCIt enrichment
    ('NCIt_Preferred_Term', 45),
    ('NCIt_Synonyms',       55),
    ('NCIt_Definition',     60),
    ('UMLS_CUI',            12),
    ('NCIm_CUI',            12),
    # COSMoS summary (aggregated per TESTCD)
    ('Has_COSMoS',          10),
    ('COSMoS_DSS_Count',    10),
    ('COSMoS_BC_ID',        14),
    ('COSMoS_BC_Name',      40),
    ('COSMoS_Category',     20),
    ('COSMoS_Subcategory',  35),
    ('COSMoS_Categories',   40),
    ('COSMoS_Parent',       35),
    ('COSMoS_Hierarchy',    50),
]

# Sheet 2 column definitions (Instrument_Specs, one row per DSS)
SPEC_COLS = [
    # Green identity — test level (from Instrument_Test_Identity or Test_Identity;
    # empty for LZZT example rows with no published CT identity)
    ('NCIt_Code',           12),
    ('TESTCD',              12),
    ('TEST',                45),
    ('Domain',              8),
    # Instrument-layer identity (C20993 / C211913 anchors)
    ('Instrument',          50),
    ('Codelist_TESTCD',     16),
    ('Instrument_NCIt_Code',            14),
    ('Instrument_NCIt_Preferred_Term',  45),
    ('Container_NCIt_Code',             14),
    ('Container_NCIt_Preferred_Term',   45),
    # Green identity — test NCIt enrichment
    ('NCIt_Preferred_Term', 45),
    ('NCIt_Synonyms',       55),
    ('NCIt_Definition',     60),
    ('UMLS_CUI',            12),
    ('NCIm_CUI',            12),
    # CT classification (per COSMoS_Instrument_Layer.md section 4)
    ('CT_Sub_Pattern',      32),
    # COSMoS BC + DSS identity (one row per DSS, no aggregation)
    ('COSMoS_BC_ID',        14),
    ('BC_Name',             40),
    ('BC_Type',             14),
    ('DS_Code',             14),
    ('DS_Name',             45),
    ('COSMoS_Category',     20),
    ('COSMoS_Subcategory',  25),
    ('Parent_Label',        35),
    ('Hierarchy_Path',      50),
    ('Categories',          40),
]

# Sheet 3 column definitions (unchanged)
GROUPING_COLS = [
    ('BC_Name',      40),
    ('BC_NCIt_Code', 12),
    ('TESTCD',       12),
    ('Domain',       8),
    ('Category',     50),
]

print('Column definitions ready.')


In [ ]:
# Styles, green/yellow colour scheme matching upstream notebooks
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

GREEN_HEADER = PatternFill('solid', fgColor='548235')
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')
# Instrument-layer palette (sibling to SDTM_Instrument_Identity.xlsx convention)
CHOCOLATE_HEADER = PatternFill('solid', fgColor='5B3A29')  # C20993 instrument identity
COPPER_HEADER    = PatternFill('solid', fgColor='CC7A3E')  # C211913 container / subscale

COSMOS_YES = PatternFill('solid', fgColor='FFFCE8')
COSMOS_NO = PatternFill('solid', fgColor='F2F2F2')


def write_sheet(ws, df, col_defs, header_fill=None):
    """Write dataframe to worksheet with formatting."""
    col_names = [c[0] for c in col_defs]
    fill = header_fill or GREEN_HEADER

    for ci, name in enumerate(col_names, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = fill
        cell.alignment = WRAP

    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(col_names, 1):
            val = row.get(name, '')
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP

    for ci, (name, width) in enumerate(col_defs, 1):
        ws.column_dimensions[get_column_letter(ci)].width = width

    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(col_names))}1'


print('Writer functions ready.')


In [ ]:
wb = Workbook()

# README
ws_readme = wb.active
ws_readme.title = 'README'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

n_per_inst = (specs['CT_Sub_Pattern'] == 'Per-instrument codelist').sum()
n_rstestcd = (specs['CT_Sub_Pattern'] == 'Domain-level RSTESTCD').sum()
n_lzzt = (specs['CT_Sub_Pattern'] == 'Not in published CT (LZZT example)').sum()

readme_lines = [
    ('Instrument Findings, Question-Level Reference', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    (f'Green source 1: {GREEN_FILE.name}  (per-instrument codelists, test grain)', readme_font),
    (f'Green source 2: {GREEN_INSTRUMENT_FILE.name}  (codelist grain, C20993 / C211913 anchors)', readme_font),
    (f'Green source 3: {GREEN_RS_FILE.name}  (RS subset, domain-level RSTESTCD)', readme_font),
    (f'Yellow source: {YELLOW_FILE.name}', readme_font),
    (f'Domain metadata: {DOMAIN_META_FILE.name}', readme_font),
    ('Notebook: Instrument_Findings.ipynb (sdtm-findings track)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    (f'Instrument domains: {" ".join(sorted(INSTRUMENT_DOMAINS))}', readme_font),
    ('Scope from COSMoS behavioural analysis. See COSMoS_Instrument_Layer.md', readme_font),
    ('and COSMoS_Behavioural_Analysis.md for the classification rationale.', readme_font),
    ('', None),
    (f'Test_Identity:       {len(test_codes):,} rows (one per TESTCD)', readme_font),
    (f'Instrument_Specs:    {len(specs):,} rows (one per Dataset Specialization)', readme_font),
    (f'Instrument_Grouping: {len(groupings):,} BC-to-Category edges', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Test_Identity, one row per TESTCD. Green identity columns from', readme_font),
    ('  SDTM_Instrument_Test_Identity (per-instrument codelists) and the RS subset', readme_font),
    ('  of SDTM_Test_Identity (domain-level RSTESTCD codelist), joined with', readme_font),
    ('  COSMoS BC summary aggregated per TESTCD where coverage exists.', readme_font),
    ('  COSMoS_DSS_Count signals fan-out: 1 for per-instrument codelist rows,', readme_font),
    ('  2 or 3 for the four multi-framework RSTESTCDs (drill into Sheet 2).', readme_font),
    ('Instrument_Specs, one row per Dataset Specialization. Yellow-driven;', readme_font),
    ('  green NCIt identity attached per row. Multi-framework TESTCDs produce', readme_font),
    ('  one row per criteria framework (RECIST 1.1, LUGANO CLASSIFICATION, RANO).', readme_font),
    ('  CT_Sub_Pattern flags the three CT codelist sub-patterns.', readme_font),
    ('Instrument_Grouping, normalised edge list of Categories tokens.', readme_font),
    ('  One row per (BC, Category). Same content as the multi-value', readme_font),
    ('  Categories columns on Sheet 1 and Sheet 2, exposed for direct querying.', readme_font),
    ('', None),
    ('ROW GRAIN', section_font),
    ('Two-sheet structure parallel to Specimen_Findings.ipynb.', readme_font),
    ('Sheet 1 (Test_Identity) is TESTCD-grain: every test code that could', readme_font),
    ('  plausibly be instrument-shaped, with COSMoS BC coverage as a column.', readme_font),
    ('  Multi-framework RSTESTCDs collapse to one row with semicolon-joined', readme_font),
    ('  framework values; COSMoS_DSS_Count points the user at Sheet 2.', readme_font),
    ('Sheet 2 (Instrument_Specs) is DSS-grain: one row per Dataset Specialization,', readme_font),
    ('  preserving the framework distinction. OVRLRESP and TRGRESP each produce', readme_font),
    ('  3 rows; NEWLIND and NTRGRESP each produce 2 rows. The framework is', readme_font),
    ('  recorded in COSMoS_Subcategory (--SCAT) and disambiguates the rows.', readme_font),
    ('', None),
    ('CT SUB-PATTERN', section_font),
    ('On Sheet 2, each row is classified into one of three CT codelist', readme_font),
    ('sub-patterns per COSMoS_Instrument_Layer.md section 4:', readme_font),
    (f'  Per-instrument codelist:           {n_per_inst} rows', readme_font),
    ('    Each instrument has its own non-extensible CT codelist (ADCTC, AIMSTC,', readme_font),
    ('    CESTC, etc.). TESTCD = DS_Code 1:1. Most of QS, all of FT, most of RS.', readme_font),
    (f'  Domain-level RSTESTCD:             {n_rstestcd} rows', readme_font),
    ('    Extensible RSTESTCD codelist plus criteria framework in --SCAT.', readme_font),
    ('    Same TESTCD can appear in multiple Dataset Specializations, one per', readme_font),
    ('    framework. Green identity comes from SDTM_Test_Identity, not', readme_font),
    ('    SDTM_Instrument_Test_Identity, so Instrument and Codelist_TESTCD are empty.', readme_font),
    (f'  Not in published CT (LZZT example): {n_lzzt} rows', readme_font),
    ('    TTS Acceptability Survey question containers from the LZZT example', readme_font),
    ('    study. TESTCDs are not in any published SDTM CT codelist; flagged', readme_font),
    ('    as a COSMoS data quality observation. Green identity columns empty.', readme_font),
    ('', None),
    ('TEST_IDENTITY COLUMNS (green identity)', section_font),
    ('NCIt_Code, NCIt C-code for the test concept', readme_font),
    ('TESTCD, SDTM short test code (submission value)', readme_font),
    ('TEST, SDTM full test name (submission value)', readme_font),
    ('Domain, SDTM domain (QS, FT or RS)', readme_font),
    ('Instrument, instrument family name (Instrument_Test_Identity rows only)', readme_font),
    ('Codelist_TESTCD, parent codelist short name (Instrument_Test_Identity rows only)', readme_font),
    ('Instrument_NCIt_Code, NCIt C-code for the instrument concept (C20993 branch; UNMATCHED rows empty)', readme_font),
    ('Instrument_NCIt_Preferred_Term, preferred term for the instrument concept', readme_font),
    ('Container_NCIt_Code, NCIt C-code for the question container (C211913 branch; 1:1 structural mapping only)', readme_font),
    ('Container_NCIt_Preferred_Term, preferred term for the question container', readme_font),
    ('NCIt_Preferred_Term, preferred term from SDTM Terminology', readme_font),
    ('NCIt_Synonyms, alternative names from NCIt (semicolon-separated)', readme_font),
    ('NCIt_Definition, definition from NCIt Thesaurus', readme_font),
    ('UMLS_CUI, Standard UMLS CUI for crosslinking', readme_font),
    ('NCIm_CUI, NCI Metathesaurus CUI (NCI-internal)', readme_font),
    ('', None),
    ('TEST_IDENTITY COLUMNS (COSMoS summary, aggregated per TESTCD)', section_font),
    ('Has_COSMoS, Yes if a COSMoS BC exists for this test code', readme_font),
    ('COSMoS_DSS_Count, number of Dataset Specializations sharing this TESTCD', readme_font),
    ('  (1 for per-instrument codelist, 2 or 3 for multi-framework RSTESTCDs)', readme_font),
    ('COSMoS_BC_ID, COSMoS internal BC identifier(s)', readme_font),
    ('COSMoS_BC_Name, biomedical concept name(s)', readme_font),
    ('COSMoS_Category, SDTM --CAT submission value(s) (FT only; empty for QS and RS)', readme_font),
    ('COSMoS_Subcategory, SDTM --SCAT submission value(s) (subscale, instrument family,', readme_font),
    ('  or criteria framework). For multi-framework RSTESTCDs, semicolon-joined', readme_font),
    ('  list of frameworks. See COSMoS_Instrument_Layer.md section 4.', readme_font),
    ('COSMoS_Categories, multi-value grouping tags (instrument families and others)', readme_font),
    ('COSMoS_Parent, immediate parent BC label (instrument question container)', readme_font),
    ('COSMoS_Hierarchy, full hierarchy path from root', readme_font),
    ('', None),
    ('INSTRUMENT_SPECS COLUMNS (green identity)', section_font),
    ('Same green identity columns as Test_Identity. Empty for LZZT example rows.', readme_font),
    ('', None),
    ('INSTRUMENT_SPECS COLUMNS (CT classification)', section_font),
    ('CT_Sub_Pattern, one of the three values described above', readme_font),
    ('', None),
    ('INSTRUMENT_SPECS COLUMNS (COSMoS BC + DSS identity, per-row, no aggregation)', section_font),
    ('COSMoS_BC_ID, COSMoS internal BC identifier', readme_font),
    ('BC_Name, biomedical concept name', readme_font),
    ('BC_Type, BC type from COSMoS', readme_font),
    ('DS_Code, dataset specialization mnemonic', readme_font),
    ('DS_Name, dataset specialization name (often embeds the criteria framework', readme_font),
    ('  for multi-framework rows, e.g. "Overall Response (RECIST 1.1)")', readme_font),
    ('COSMoS_Category, SDTM --CAT submission value', readme_font),
    ('COSMoS_Subcategory, SDTM --SCAT submission value', readme_font),
    ('Parent_Label, immediate parent BC label', readme_font),
    ('Hierarchy_Path, full hierarchy path from root', readme_font),
    ('Categories, multi-value grouping tags', readme_font),
    ('', None),
    ('INSTRUMENT_GROUPING COLUMNS', section_font),
    ('BC_Name, biomedical concept name (one row per Category token)', readme_font),
    ('BC_NCIt_Code, NCIt C-code for the BC', readme_font),
    ('TESTCD, SDTM submission value', readme_font),
    ('Domain, SDTM domain', readme_font),
    ('Category, single Category token (instrument family or other grouping)', readme_font),
    ('', None),
    ('IDENTIFIER NOTE', section_font),
    ('Categories tokens are labels, not stable identifiers. About 60% of distinct', readme_font),
    ('instrument categories carry no identifier and depend on string equality for', readme_font),
    ('grouping. Every instrument BC depends on at least one label-only Category in', readme_font),
    ('its grouping set. See COSMoS_Instrument_Layer.md section 6 for the analysis.', readme_font),
    ('', None),
    ('DEFERRED', section_font),
    ('NCIt C211913 reference rows for the materialization gap (345 NCIt children of', readme_font),
    ('CDISC QRS Instruments Questions not yet materialized as COSMoS BCs) are not', readme_font),
    ('included in this v1. Pending an upstream artifact decision: either the probe', readme_font),
    ('notebook persists C211913_children.csv into cosmos-bc-dss/interim, or this', readme_font),
    ('notebook reads Thesaurus.txt directly. See COSMoS_Instrument_Layer.md section 5.3.', readme_font),
    ('Instrument-first or framework-first summary view also deferred (section 5.2).', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('Consumer track output. Column names use COSMoS vocabulary for traceability.', readme_font),
    ('Not an official CDISC product. Sources: NCI EVS + COSMoS public exports.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_readme.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_readme.column_dimensions['A'].width = 100
print(f'README: {len(readme_lines)} lines')


In [ ]:
# Test_Identity sheet (one row per TESTCD)
ws_t = wb.create_sheet('Test_Identity')
t_col_names = [c[0] for c in TESTCODE_COLS]
write_sheet(ws_t, test_codes[t_col_names], TESTCODE_COLS, header_fill=GREEN_HEADER)

# Override instrument-layer NCIt anchor columns with chocolate / copper headers
for name, fill in [
    ('Instrument_NCIt_Code',           CHOCOLATE_HEADER),
    ('Instrument_NCIt_Preferred_Term', CHOCOLATE_HEADER),
    ('Container_NCIt_Code',            COPPER_HEADER),
    ('Container_NCIt_Preferred_Term',  COPPER_HEADER),
]:
    ci = t_col_names.index(name) + 1
    ws_t.cell(row=1, column=ci).fill = fill

# Override COSMoS summary columns (last 9) with yellow headers
cosmos_start_t = len(TESTCODE_COLS) - 9 + 1  # 1-indexed
for ci in range(cosmos_start_t, len(TESTCODE_COLS) + 1):
    ws_t.cell(row=1, column=ci).fill = YELLOW_HEADER

# Conditional fill on Has_COSMoS column (the long-defined COSMOS_YES/COSMOS_NO
# constants from Cell 17 are used here)
has_cosmos_ci = t_col_names.index('Has_COSMoS') + 1
for ri in range(2, len(test_codes) + 2):
    val = ws_t.cell(row=ri, column=has_cosmos_ci).value
    if val == 'Yes':
        ws_t.cell(row=ri, column=has_cosmos_ci).fill = COSMOS_YES
    elif val == 'No':
        ws_t.cell(row=ri, column=has_cosmos_ci).fill = COSMOS_NO

print(f'Test_Identity: {len(test_codes):,} rows')


In [ ]:
# Instrument_Specs sheet (one row per Dataset Specialization)
ws_q = wb.create_sheet('Instrument_Specs')
q_col_names = [c[0] for c in SPEC_COLS]
write_sheet(ws_q, specs[q_col_names], SPEC_COLS, header_fill=GREEN_HEADER)

# Override instrument-layer NCIt anchor columns with chocolate / copper headers
for name, fill in [
    ('Instrument_NCIt_Code',           CHOCOLATE_HEADER),
    ('Instrument_NCIt_Preferred_Term', CHOCOLATE_HEADER),
    ('Container_NCIt_Code',            COPPER_HEADER),
    ('Container_NCIt_Preferred_Term',  COPPER_HEADER),
]:
    ci = q_col_names.index(name) + 1
    ws_q.cell(row=1, column=ci).fill = fill

# Override COSMoS columns (last 10) with yellow headers.
# Column 12 (CT_Sub_Pattern) is classification metadata derived from the
# join, kept under the green header by convention.
cosmos_start = len(SPEC_COLS) - 10 + 1  # 1-indexed
for ci in range(cosmos_start, len(SPEC_COLS) + 1):
    ws_q.cell(row=1, column=ci).fill = YELLOW_HEADER

# Conditional fill on CT_Sub_Pattern column
CT_PATTERN_FILLS = {
    'Per-instrument codelist':            PatternFill('solid', fgColor='E8F5E9'),
    'Domain-level RSTESTCD':              PatternFill('solid', fgColor='FFF8E1'),
    'Not in published CT (LZZT example)': PatternFill('solid', fgColor='F2F2F2'),
}
ct_ci = q_col_names.index('CT_Sub_Pattern') + 1
for ri in range(2, len(specs) + 2):
    val = ws_q.cell(row=ri, column=ct_ci).value
    if val in CT_PATTERN_FILLS:
        ws_q.cell(row=ri, column=ct_ci).fill = CT_PATTERN_FILLS[val]

print(f'Instrument_Specs: {len(specs):,} rows')


In [ ]:
# Instrument_Grouping sheet
ws_g = wb.create_sheet('Instrument_Grouping')
g_col_names = [c[0] for c in GROUPING_COLS]
write_sheet(ws_g, groupings[g_col_names], GROUPING_COLS, header_fill=YELLOW_HEADER)

# Link columns (BC_NCIt_Code, TESTCD) get green headers
for ci in [2, 3]:
    ws_g.cell(row=1, column=ci).fill = GREEN_HEADER

print(f'Instrument_Grouping: {len(groupings):,} rows')

## 9. Save and Summary


In [ ]:
wb.save(OUTPUT_FILE)
print(f'Output: {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB')
print()
print('=== Summary ===')
print(f'Instrument domains:           {sorted(INSTRUMENT_DOMAINS)}')
print()
print(f'Test_Identity sheet:          {len(test_codes):,} rows  (one per TESTCD)')
print(f'  With COSMoS BC coverage:    {n_with:,}')
print(f'  Without COSMoS BC coverage: {n_without:,}')
print()
print(f'Instrument_Specs sheet:       {len(specs):,} rows  (one per Dataset Specialization)')
print(f'  Per-instrument codelist:    {(specs["CT_Sub_Pattern"]=="Per-instrument codelist").sum()}')
print(f'  Domain-level RSTESTCD:      {(specs["CT_Sub_Pattern"]=="Domain-level RSTESTCD").sum()}')
print(f'  Not in published CT (LZZT): {(specs["CT_Sub_Pattern"]=="Not in published CT (LZZT example)").sum()}')
print()
print(f'Instrument_Grouping sheet:    {len(groupings):,} rows  (BC-to-Category edges)')
print(f'  Distinct BCs:               {groupings["BC_Name"].nunique()}')
print(f'  Distinct categories:        {groupings["Category"].nunique()}')
print()
print('Reference file ready.')
